# 🎵 Suno Zara — free GPU music worker (Google Colab or Kaggle)

This notebook runs the **ACE-Step 1.5** AI music model on a free GPU. It picks up the songs requested in the
Suno Zara app, makes them **with real singing**, and saves them to your app.

### On Google Colab (no phone number needed)
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **File → Save a copy in Drive** (so your settings are kept; then always open *that* copy)
3. **Runtime → Run all**. Boxes will appear asking for your 3 values (the same ones you gave Render): paste each one and press **Enter**
4. **Keep this page open** while songs are made

### On Kaggle
Settings → **GPU T4 x2** and **Internet On**; Add-ons → **Secrets**: `DATABASE_URL`, `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`; then **Run all**.

The first song is ready about **10–15 minutes** after starting (installing + downloading the AI model).

In [ ]:
# ⚙️ Settings
# Paste your values between the quotes (same as Render). On Kaggle you can use Add-ons → Secrets instead.
DATABASE_URL = ""
SUPABASE_URL = ""
SUPABASE_SERVICE_ROLE_KEY = ""
GROQ_API_KEY = ""      # optional: free key from console.groq.com for AI-written lyrics

RUN_HOURS = 3          # how long to keep making songs
SONG_SECONDS = 180     # length of new songs (3 minutes)
REPO = "https://github.com/cockroachbankersparty01-gif/suno-zara"

In [ ]:
# 1️⃣ Check GPU, internet and settings
import os, subprocess, sys, time, json, urllib.request

ON_KAGGLE = os.path.isdir("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content" if os.path.isdir("/content") else os.getcwd()
print("Platform:", "Kaggle" if ON_KAGGLE else "Google Colab" if ON_COLAB else "other", "| folder:", WORK)

def fail(msg):
    raise SystemExit("\n❌ " + msg + "\n")

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    fail("No GPU. " + ("Settings → Accelerator → 'GPU T4 x2'" if ON_KAGGLE else "Runtime → Change runtime type → 'T4 GPU' → Save")
         + ", then run everything again. (On Colab: if no GPU is available right now, try again later.)")
print("✅ GPU:", gpu.stdout.strip().splitlines()[0])

try:
    urllib.request.urlopen("https://github.com", timeout=15)
    print("✅ Internet is on")
except Exception:
    fail("No internet. On Kaggle: Settings → Internet → On.")

secrets = {"DATABASE_URL": DATABASE_URL, "SUPABASE_URL": SUPABASE_URL,
           "SUPABASE_SERVICE_ROLE_KEY": SUPABASE_SERVICE_ROLE_KEY, "GROQ_API_KEY": GROQ_API_KEY}
def from_store(name):
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        return ""
for name in secrets:
    if not secrets[name].strip():
        secrets[name] = (from_store(name) or "").strip()
    secrets[name] = secrets[name].strip().strip('"').strip("'")
    if secrets[name].upper().startswith(name + " ="):
        secrets[name] = secrets[name].split("=", 1)[1].strip()

# Still empty? Ask for them in a box (easy on a phone: just paste and press Enter).
if not ON_KAGGLE:
    from getpass import getpass
    for name in ["DATABASE_URL", "SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"]:
        if not secrets[name]:
            print(f"👉 Paste your {name} in the box below, then press Enter:")
            value = getpass(f"{name}: ") if name != "SUPABASE_URL" else input(f"{name}: ")
            value = value.strip().strip('"').strip("'")
            if value.upper().startswith(name + " ="):
                value = value.split("=", 1)[1].strip()
            secrets[name] = value

missing = [n for n in ["DATABASE_URL", "SUPABASE_URL", "SUPABASE_SERVICE_ROLE_KEY"] if not secrets[n]]
if missing:
    fail("Missing: " + ", ".join(missing) + ". Paste them into the ⚙️ Settings cell above (between the quotes), then run all again.")
if not secrets["DATABASE_URL"].startswith("postgres"):
    fail("DATABASE_URL should start with postgresql:// (copy it again from your notes).")
if "[" in secrets["DATABASE_URL"]:
    fail("DATABASE_URL still has [YOUR-PASSWORD] in it. Replace it, including the brackets, with your database password.")
if not secrets["SUPABASE_URL"].startswith("https://"):
    fail("SUPABASE_URL should look like https://xxxx.supabase.co")
if not secrets["SUPABASE_SERVICE_ROLE_KEY"].startswith(("sb_secret_", "eyJ")):
    fail("SUPABASE_SERVICE_ROLE_KEY should be your secret key (starts with sb_secret_), not the publishable one.")
print("✅ Settings look good")

In [ ]:
# 2️⃣ Install ACE-Step 1.5 (~3–5 min)
import os, subprocess
ACE = f"{WORK}/ACE-Step-1.5"
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"❌ Failed: {cmd}")
    return r.stdout
# ffmpeg first: ACE-Step needs it to save MP3s
if subprocess.run("which ffmpeg", shell=True).returncode != 0:
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
sh("pip install -q uv")
if not os.path.isdir(ACE):
    sh(f"git clone --depth 1 https://github.com/ACE-Step/ACE-Step-1.5.git {ACE}")
print(sh(f"cd {ACE} && uv sync 2>&1 | tail -3"))
print("✅ ACE-Step installed")

In [ ]:
# 3️⃣ Start the ACE-Step music server on the GPU
import subprocess, time, urllib.request, os, json
ace_env = dict(os.environ,
    CUDA_VISIBLE_DEVICES="0",
    ACESTEP_DEVICE="cuda",
    ACESTEP_CONFIG_PATH="acestep-v15-turbo",
    ACESTEP_LM_BACKEND="pt",               # vLLM may not support the T4
    ACESTEP_INIT_LLM="false",              # the extra "thinking" LM isn't needed and saves memory
    ACESTEP_USE_FLASH_ATTENTION="false",   # T4 is too old for flash-attention
    ACESTEP_API_HOST="127.0.0.1",
    ACESTEP_API_PORT="8001",
)
# Colab/Jupyter set these for the notebook itself; ACE-Step runs in its own environment and must not inherit them.
for k in ("MPLBACKEND", "PYTHONPATH", "PYTHONHOME"):
    ace_env.pop(k, None)
ace_env["MPLBACKEND"] = "Agg"
ace_log = open(f"{WORK}/acestep.log", "w")
ace = subprocess.Popen("uv run acestep-api", shell=True, cwd=ACE, env=ace_env, stdout=ace_log, stderr=subprocess.STDOUT)

def health():
    try:
        return json.load(urllib.request.urlopen("http://127.0.0.1:8001/health", timeout=5))
    except Exception:
        return None

t0 = time.time()
while not health():
    if ace.poll() is not None:
        print(open(f"{WORK}/acestep.log").read()[-4000:])
        raise SystemExit("❌ ACE-Step stopped while starting (log above).")
    if time.time() - t0 > 1200:
        print(open(f"{WORK}/acestep.log").read()[-4000:])
        raise SystemExit("❌ ACE-Step didn't start in 20 minutes (log above).")
    time.sleep(10)
print(f"✅ ACE-Step running ({int(time.time()-t0)}s)")

In [ ]:
# 4️⃣ Warm-up: make a 10-second test clip (downloads the model the first time, ~3–8 min)
import urllib.request, urllib.parse, json, time
def post(path, data, form=False):
    if form:
        body = urllib.parse.urlencode(data).encode()
        req = urllib.request.Request("http://127.0.0.1:8001" + path, data=body)
    else:
        req = urllib.request.Request("http://127.0.0.1:8001" + path, data=json.dumps(data).encode(), headers={"content-type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=60))

t0 = time.time()
task = post("/release_task", {"prompt": "happy acoustic guitar", "lyrics": "[Instrumental]", "audio_duration": "10",
                              "batch_size": "1", "audio_format": "mp3", "task_type": "text2music"}, form=True)["data"]["task_id"]
while True:
    item = post("/query_result", {"task_id_list": [task]})["data"][0]
    if item["status"] == 1:
        print(f"✅ Test clip made in {int(time.time()-t0)}s. The GPU is ready.")
        break
    if item["status"] == 2:
        print(open(f"{WORK}/acestep.log").read()[-4000:])
        raise SystemExit("❌ Test clip failed: " + str(item.get("result")))
    if time.time() - t0 > 1500:
        raise SystemExit("❌ Test clip took too long")
    time.sleep(5)

In [ ]:
# 5️⃣ Install the Suno Zara worker (~1–2 min)
import os, subprocess
def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"❌ Failed: {cmd}")
    return r.stdout
NODE = f"{WORK}/node"
if not os.path.exists(f"{NODE}/bin/node"):
    sh(f"cd {WORK} && curl -fsSL https://nodejs.org/dist/v22.12.0/node-v22.12.0-linux-x64.tar.xz -o node.tar.xz "
       "&& mkdir -p node && tar -xJf node.tar.xz -C node --strip-components=1 && rm node.tar.xz")
os.environ["PATH"] = f"{NODE}/bin:" + os.environ["PATH"]
if subprocess.run("which ffmpeg", shell=True).returncode != 0:
    sh("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
APP = f"{WORK}/suno-zara"
if os.path.isdir(APP):
    sh(f"cd {APP} && git pull -q")
else:
    sh(f"git clone --depth 1 {REPO} {APP}")
print(sh(f"cd {APP}/backend && npm install --no-audit --no-fund 2>&1 | tail -2 && npm run build 2>&1 | tail -2"))
print("✅ Worker ready")

In [ ]:
# 6️⃣ Make songs! Leave this running. New requests from the app are picked up automatically.
import subprocess, os, time, signal
w_env = dict(os.environ,
    DATABASE_URL=secrets["DATABASE_URL"],
    SUPABASE_URL=secrets["SUPABASE_URL"],
    SUPABASE_SERVICE_ROLE_KEY=secrets["SUPABASE_SERVICE_ROLE_KEY"],
    AUTH_MODE="supabase", STORAGE_DRIVER="supabase", STORAGE_BUCKET="media",
    MUSIC_PROVIDER="acestep", ACESTEP_URL="http://127.0.0.1:8001",
    IMAGE_PROVIDER="pollinations",
    DEFAULT_DURATION_SEC=str(SONG_SECONDS), MAX_DURATION_SEC="360",
    DB_POOL_MAX="3", WORKER_CONCURRENCY="1",
)
w_env.pop("MPLBACKEND", None)
if secrets.get("GROQ_API_KEY"):
    w_env.update(LYRICS_PROVIDER="openai", LLM_BASE_URL="https://api.groq.com/openai/v1",
                 LLM_API_KEY=secrets["GROQ_API_KEY"], LLM_MODEL="llama-3.3-70b-versatile")
else:
    w_env.update(LYRICS_PROVIDER="mock")

worker = subprocess.Popen("node dist/worker.js", shell=True, cwd=f"{APP}/backend", env=w_env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
print(f"🎵 Making songs for {RUN_HOURS} hours. Open the app and create a song!\n")
import threading
def times_up():
    print("⏰ Time's up: finishing the current song, then stopping.", flush=True)
    worker.send_signal(signal.SIGTERM)
timer = threading.Timer(RUN_HOURS * 3600, times_up)
timer.daemon = True
timer.start()
try:
    for line in worker.stdout:          # ends when the worker exits
        print(time.strftime("%H:%M"), line, end="", flush=True)
    worker.wait()
finally:
    timer.cancel()
    if worker.poll() is None:
        worker.terminate()
    ace.terminate()
print("👋 Worker stopped. Songs requested from now on wait until you run this notebook again.")